# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I am choosing **Lane 2: Refresh / Content Opportunity Scoring** as my provisional lane.

This lane seems worth the next seven weeks because the starter data is already organized at the page level and contains the kinds of signals a content reviewer would use before deciding what to review: impressions, clicks, CTR, average position, sessions, engagement, content age, freshness, and trend direction. The output would not be "a model for its own sake." The useful product would be a ranked review queue that helps a human decide which pages deserve refresh, metadata review, expansion, protection, pruning, or monitoring first.

I am choosing this over freestyle for now because it has a clear decision, a clear action, and enough starter data to build an honest baseline before trying a model. I can still adjust the lane by Week 4 if the later warehouse data shows a better question.


In [1]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(DATA_PATH)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Pseudonymized clients: {df['client_id'].nunique():,}")
print(f"One row per content item? {df['content_id'].is_unique}")


Rows: 30,000
Columns: 44
Pseudonymized clients: 32
One row per content item? True


## 2. The question: decision, action, cost of a wrong call

**Research question:** Which pages should a content reviewer inspect first for refresh or CTR/engagement review, using observable search, traffic, freshness, and content signals from the starter data?

**Decision improved:** The decision is not "can I predict decline?" The decision is **which page enters the review queue first** when reviewer time is limited.

**Unit of analysis:** One row is one pseudonymized content item, or page. I will not use `content_id` or `client_id` as model features; they are only for grouping, deduplication, and validation design.

**Output:** A ranked page-level opportunity score with reason codes such as declining with demand, low CTR while visible, stale visible page, page-one decay risk, or weak engagement.

**Action someone could take:** A content strategist, SEO analyst, or editor could review the top-ranked pages and decide whether to update content, improve title/meta alignment, expand thin sections, protect a strong page, prune a weak page, or monitor instead of acting.

**Cost of a wrong recommendation:** A false positive wastes review and editing time on a page that may not need work. A false negative is more costly if a page with real demand keeps losing visibility, clicks, or engagement while the team spends time elsewhere. A bad recommendation can also cause harmful edits to a page that was already performing well, so the output must include reason codes and human review rather than automatic changes.

**Why data or ML can help:** A plain rule can catch obvious cases, but page priority depends on several signals at once: demand, position, CTR, trend, freshness, content depth, and engagement. The useful ML question is whether a learned ranking can improve the top of the review queue compared with transparent rules, while still staying explainable enough for a reviewer to trust.

**Provisional task type and metric:** This is mainly a **ranking / scoring** task. The metric should match review capacity, so I will use top-K metrics such as precision@20 or precision@50, plus manual review of high-ranked examples. The starter label `trend_direction == "down"` is only a proxy; for the capstone I should prefer a future-window outcome from the warehouse if the data contract supports it.


In [2]:
# Starter rule-style candidate counts for the refresh/opportunity lane.
# Rate columns such as ctr are x100 percentages, so ctr < 0.5 means below 0.5%.

candidate_flags = pd.DataFrame(
    {
        "declining_with_demand": (df["trend_direction"].eq("down") & df["impressions_90d"].ge(100)),
        "low_ctr_visible_page": (
            df["impressions_90d"].ge(500)
            & df["avg_position"].gt(0)
            & df["avg_position"].le(20)
            & df["ctr"].lt(0.5)
        ),
        "page_one_decay_risk": (
            df["avg_position"].gt(0)
            & df["avg_position"].le(10)
            & df["content_age_days"].ge(180)
        ),
        "engagement_review_candidate": (
            df["sessions_90d"].ge(30)
            & (df["engagement_rate"].lt(30) | df["scroll_rate"].lt(30))
        ),
    }
)

candidate_summary = (
    candidate_flags.sum()
    .rename("pages")
    .to_frame()
    .assign(percent_of_dataset=lambda x: (100 * x["pages"] / len(df)).round(1))
)

candidate_summary


,pages,percent_of_dataset
declining_with_demand,13152,43.8
low_ctr_visible_page,9759,32.5
page_one_decay_risk,7076,23.6
engagement_review_candidate,7113,23.7


## 3. Quick look at the data (2-3 real numbers)

The starter CSV has **30,000 pages**, **44 columns**, and **32 pseudonymized clients**. The unit of analysis fits the lane because each row is one content item/page.

A few numbers make the refresh/opportunity lane look worth exploring:

- **16,262 pages (54.2%)** are marked `down` by the starter trend proxy. That is not a final capstone target, but it shows there are many pages with negative movement to triage.
- **16,726 pages (55.8%)** have at least 500 impressions in the trailing 90-day window, so many pages have enough search visibility for prioritization to matter.
- **9,759 pages (32.5%)** are visible pages with average position 1-20, at least 500 impressions, and CTR below 0.5%. Since CTR is stored as a x100 percentage, `0.5` means 0.5%, not 50%.

Those numbers suggest a ranked queue is more useful than manually scanning every page. They also show why careful thresholds matter: if the queue is too broad, it will overwhelm a reviewer.


In [3]:
summary = pd.Series(
    {
        "rows_pages": len(df),
        "columns": df.shape[1],
        "pseudonymized_clients": df["client_id"].nunique(),
        "declining_pages": df["trend_direction"].eq("down").sum(),
        "declining_pct": round(100 * df["trend_direction"].eq("down").mean(), 1),
        "pages_with_500plus_impressions": df["impressions_90d"].ge(500).sum(),
        "pages_with_500plus_impressions_pct": round(100 * df["impressions_90d"].ge(500).mean(), 1),
        "low_ctr_visible_pages": candidate_flags["low_ctr_visible_page"].sum(),
        "low_ctr_visible_pages_pct": round(100 * candidate_flags["low_ctr_visible_page"].mean(), 1),
        "avg_position_zero_no_data_rows": df["avg_position"].eq(0).sum(),
    }
)

summary.to_frame("value")


,value
rows_pages,30000.0
columns,44.0
pseudonymized_clients,32.0
declining_pages,16262.0
declining_pct,54.2
pages_with_500plus_impressions,16726.0
pages_with_500plus_impressions_pct,55.8
low_ctr_visible_pages,9759.0
low_ctr_visible_pages_pct,32.5
avg_position_zero_no_data_rows,1205.0


## 4. Careful words: what I can and can't claim

What I can claim from this Week 1 notebook is limited: the starter data contains enough page-level variation to make **refresh/opportunity ranking** a reasonable provisional lane. I can say the data supports decision-support work because there are many visible pages, many pages with decline-like movement, and many visible low-CTR candidates.

What I cannot claim yet:

- I cannot claim that refreshing a page will cause recovery. That would require an experiment or a stronger causal design.
- I cannot claim that low CTR is caused by a bad title or meta description. It could reflect intent mismatch, SERP layout, seasonality, brand demand, or noise.
- I cannot treat `trend_direction` or `trend_pct` as ordinary features if I use the starter declining label, because the label is derived from them.
- I cannot use `avg_position = 0` as rank zero; it means no position data.
- I cannot publish raw client names, URLs, queries, titles, or anything that tries to reverse the pseudonymized IDs.

The careful version of my claim is: **I am building a decision-support ranking that helps a human reviewer choose which pages to inspect first. The ranking may improve prioritization, but it does not automatically decide or prove what edit should be made.**


In [4]:
# Small leakage and gotcha checks I need to remember before modeling later.
checks = pd.Series(
    {
        "trend_direction_used_only_as_proxy_context": True,
        "trend_pct_excluded_from_future_model_features": True,
        "avg_position_zero_means_no_data_rows": int(df["avg_position"].eq(0).sum()),
        "content_id_unique": bool(df["content_id"].is_unique),
        "client_id_feature_allowed": False,
        "content_id_feature_allowed": False,
    }
)

checks.to_frame("check")


,check
trend_direction_used_only_as_proxy_context,True
trend_pct_excluded_from_future_model_features,True
avg_position_zero_means_no_data_rows,1205
content_id_unique,True
client_id_feature_allowed,False
content_id_feature_allowed,False


## Self-check

Before submitting, I checked each line honestly:

- [x] Every section above is filled with markdown thinking and code-backed numbers.
- [x] The notebook runs top to bottom with no errors.
- [x] I picked one predefined lane: **Refresh / Content Opportunity Scoring**.
- [x] I named the decision: which pages should enter the review queue first.
- [x] I named the action: human review for refresh, metadata, expansion, protection, pruning, or monitoring.
- [x] I named the cost of a wrong recommendation: wasted reviewer time, missed decline/opportunity, or harmful unnecessary edits.
- [x] I showed real numbers from `data/raw/content_refresh_anonymized.csv`.
- [x] I explained why this is not just "train a model": the goal is a ranked, explainable decision-support queue.
- [x] I used careful language and did not claim causality, Google algorithm proof, or automatic refresh success.
